# Incendiile din Domogled – Valea Cernei · analiză reproductibilă

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bunilastelian/domogled-live/blob/main/analiza.ipynb)

Acest notebook reproduce, de la zero, cifrele din raportul despre incendiile din
Parcul Național Domogled – Valea Cernei și din Parcul Natural Porțile de Fier.

**Nu ai nevoie de nimic instalat și de nicio cheie API.** Citește datele publice pe care
le colectează monitorul și le verifică în fața ta. Rulează toate celulele cu
*Runtime → Run all*, sau cu Ctrl+F9.

### Ce vei vedea

1. Câte focare **distincte** sunt, nu câte puncte de detecție
2. Intensitatea în timp (FRP, în MW) și tendința pe 24 h
3. Unde anume, față de repere cunoscute (vârfuri, cascade, localități)
4. Cât din suprafață e afectată, estimat independent din Sentinel-2
5. Meteo, indice de pericol și direcția de propagare

### Despre ce e și ce nu e acest notebook

**Ce este:** o consolă de analiză, care citește date publice și le interpretează.

**Ce nu este:** un serviciu de monitorizare permanentă. Colab (varianta gratuită)
închide sesiunea după cel mult 12 ore și mașina e ștearsă când rămâne inactivă —
Google o spune explicit în [FAQ](https://research.google.com/colaboratory/faq.html):
*„Runtimes will time out if you are idle"* și *„notebooks can run for at most 12 hours"*.
Monitorizarea continuă rulează în GitHub Actions, la 15 minute.

## 0. Pregătire

Instalăm doar ce lipsește. În Colab durează câteva secunde.

In [ ]:
import sys, subprocess
import importlib.util  # atentie: 'import importlib' NU aduce submodulul .util

NECESARE = {"pandas": "pandas", "numpy": "numpy", "folium": "folium", "requests": "requests"}
lipsa = [pkg for mod, pkg in NECESARE.items() if importlib.util.find_spec(mod) is None]
if lipsa:
    print("instalez:", " ".join(lipsa))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *lipsa], check=True)
else:
    print("toate dependințele sunt prezente")

import json
import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("gata")

## 1. Sursele de date

Datele vin de la monitorul public. Dacă rulăm din Colab, le luăm de pe site-ul publicat;
dacă rulăm local, din fișierul scris de colector.

| sursă | ce dă |
|---|---|
| NASA FIRMS (VIIRS 375 m, MODIS 1 km) | puncte de detecție termică |
| ESA Sentinel-3 SLSTR Level 2 FRP | putere radiativă, temperatură de brilianță, încredere |
| ESA Sentinel-2 L2A | benzi pentru indicele de arsură (dNBR) |
| Open-Meteo | temperatură, umiditate, vânt, precipitații |

In [ ]:
from pathlib import Path

URL_PUBLIC = "https://bunilastelian.github.io/domogled-live/state/state.json"
LOCAL = Path("live/state/state.json")

if LOCAL.exists():
    stare = json.loads(LOCAL.read_text(encoding="utf-8"))
    sursa = f"fișier local: {LOCAL}"
else:
    r = requests.get(URL_PUBLIC, timeout=45)
    r.raise_for_status()
    stare = r.json()
    sursa = URL_PUBLIC

zone = stare.get("zones") or {}
print(f"sursă: {sursa}")
print(f"zone monitorizate: {len(zone)} · cicluri de colectare: {stare.get('cycles')}")
for zid, z in zone.items():
    s = z.get("stats") or {}
    print(f"  {z.get('name', zid):<34} {len(z.get('detections') or {}):>4} detecții · "
          f"actualizat {s.get('updated', '?')}")

## 2. Câte incendii sunt, de fapt

Aici e capcana care se repetă în presă: **un focar produce zeci de puncte de detecție**,
la fiecare trecere a satelitului. Numărul de puncte nu este numărul de incendii.

Grupăm punctele care sunt la mai puțin de 2 km unul de altul (*single-linkage*): dacă două
detecții sunt apropiate, sunt considerate același focar.

In [ ]:
import math
from datetime import datetime, timezone

RAZA_KM = 2.0
MIN_PUNCTE = 3


def dist_km(a, b):
    """Distanța pe sfera terestră, formula haversine."""
    R = 6371.0
    p1, p2 = math.radians(a[0]), math.radians(b[0])
    dp = p2 - p1
    dl = math.radians(b[1] - a[1])
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))


def cand(d):
    t = str(d.get("time", "0000")).zfill(4)
    try:
        return datetime.strptime(f"{d['date']} {t[:4]}", "%Y-%m-%d %H%M").replace(tzinfo=timezone.utc)
    except (KeyError, ValueError):
        return None


def grupeaza(puncte, raza_km=RAZA_KM):
    """Single-linkage cu union-find, pe grilă, ca să nu comparăm toate perechile."""
    n = len(puncte)
    parinte = list(range(n))

    def radacina(a):
        while parinte[a] != a:
            parinte[a] = parinte[parinte[a]]
            a = parinte[a]
        return a

    celula = raza_km / 111.0
    grila = {}
    for i, p in enumerate(puncte):
        grila.setdefault((int(p["lat"] / celula), int(p["lon"] / celula)), []).append(i)

    for i, p in enumerate(puncte):
        cy, cx = int(p["lat"] / celula), int(p["lon"] / celula)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                for j in grila.get((cy + dy, cx + dx), ()):
                    if j <= i:
                        continue
                    q = puncte[j]
                    if dist_km((p["lat"], p["lon"]), (q["lat"], q["lon"])) <= raza_km:
                        ra, rb = radacina(i), radacina(j)
                        if ra != rb:
                            parinte[rb] = ra

    grupuri = {}
    for i in range(n):
        grupuri.setdefault(radacina(i), []).append(puncte[i])  # punctul, nu indicele
    return sorted(grupuri.values(), key=len, reverse=True)


def descrie(grup, z, acum=None):
    acum = acum or datetime.now(timezone.utc)
    lat = [d["lat"] for d in grup]
    lon = [d["lon"] for d in grup]
    clat, clon = sum(lat) / len(lat), sum(lon) / len(lon)
    timpi = [t for t in (cand(d) for d in grup) if t]

    def frp_intre(lo, hi):
        s = 0.0
        for d in grup:
            t = cand(d)
            if t and lo <= (acum - t).total_seconds() / 3600 < hi:
                s += d.get("frp") or 0
        return s

    f24, f48 = frp_intre(0, 24), frp_intre(24, 48)
    if f24 > f48 * 1.25:
        tendinta = "în creștere"
    elif f24 < f48 * 0.75:
        tendinta = "în scădere"
    else:
        tendinta = "staționar"

    aproape, dmin = None, None
    for nume, (plat, plon) in (z.get("places") or {}).items():
        dd = dist_km((clat, clon), (plat, plon))
        if dmin is None or dd < dmin:
            aproape, dmin = nume, dd

    intindere = max((dist_km((grup[i]["lat"], grup[i]["lon"]), (grup[j]["lat"], grup[j]["lon"]))
                     for i in range(len(grup)) for j in range(i + 1, len(grup))), default=0.0)

    return {
        "lat": round(clat, 5), "lon": round(clon, 5), "puncte": len(grup),
        "zile": len({d["date"] for d in grup}),
        "prima": min(timpi).strftime("%Y-%m-%d %H:%M") if timpi else None,
        "ultima": max(timpi).strftime("%Y-%m-%d %H:%M") if timpi else None,
        "FRP_max_MW": round(max((d.get("frp") or 0) for d in grup), 1),
        "FRP_24h_MW": round(f24, 1), "FRP_24h_anterior_MW": round(f48, 1),
        "tendinta": tendinta, "intindere_km": round(intindere, 2),
        "NASA": sum(1 for d in grup if d.get("src") == "NASA"),
        "ESA": sum(1 for d in grup if d.get("src") == "ESA"),
        "cel_mai_aproape_reper": aproape, "distanta_km": round(dmin, 2) if dmin else None,
    }


toate_focarele = {}
# reperele (orase, varfuri) sunt in zone_list, nu in starea fiecarei zone
meta = {x["id"]: x for x in (stare.get("zone_list") or [])}
for zid, z in zone.items():
    zz = {**z, "places": (meta.get(zid) or {}).get("places") or {}}
    puncte = list((z.get("detections") or {}).values())
    gasite = [descrie(g, zz) for g in grupeaza(puncte) if len(g) >= MIN_PUNCTE]
    gasite.sort(key=lambda f: f["FRP_24h_MW"], reverse=True)
    toate_focarele[zid] = gasite
    print(f"\n=== {z.get('name', zid)} — {len(gasite)} focare ===")
    if gasite:
        print(pd.DataFrame(gasite).to_string(index=False))
    else:
        print("  niciun focar cu suficiente detecții")

## 3. Intensitatea, în timp

FRP = *Fire Radiative Power*, puterea radiată de foc, în megawați. Sunt **măsurători
instantanee** la momentul trecerii satelitului, nu energii totale. Comparația dintre
zile arată tendința.

In [ ]:
import matplotlib.pyplot as plt

fig, axe = plt.subplots(1, len(zone), figsize=(8 * len(zone), 4.2), squeeze=False)
for ax, (zid, z) in zip(axe[0], zone.items()):
    zile = (z.get("stats") or {}).get("days") or []
    if not zile:
        ax.set_title(f"{z.get('name', zid)} — fără date")
        continue
    d = pd.DataFrame(zile)
    x = np.arange(len(d))
    ax.bar(x, d["frp_nasa"], 0.62, label="NASA FIRMS", color="#f76b15")
    ax.bar(x, d["frp_esa"], 0.62, bottom=d["frp_nasa"], label="ESA Sentinel-3", color="#3b82f6")
    ax.set_xticks(x)
    ax.set_xticklabels(d["date"].str.slice(5), rotation=45, ha="right")
    ax.set_ylabel("FRP cumulat (MW)")
    ax.set_title(z.get("name", zid))
    ax.grid(axis="y", alpha=0.25, ls=":")
    for i, n in enumerate(d["count"]):
        ax.text(i, (d["frp_nasa"][i] + d["frp_esa"][i]) + 20, str(n), ha="center", fontsize=8)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Numărul de sub fiecare bară = câte detecții au fost în ziua respectivă.")

## 4. Unde, exact

Harta arată detecțiile colorate după vechime, conturul parcului și suprafața arsă estimată.
Culoarea nu e decorativă: roșu = ultimele 6 ore, portocaliu = 24 h, galben = 3 zile.

In [ ]:
import folium
from IPython.display import display


def harta(zid, z, max_puncte=500):
    aoi = z.get("aoi") or {}
    centru = aoi.get("center") or [44.8982, 22.4785]
    m = folium.Map(location=centru, zoom_start=aoi.get("zoom", 12), tiles=None)
    folium.TileLayer(
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri, Maxar", name="Satelit").add_to(m)
    folium.TileLayer("OpenStreetMap", name="Hartă").add_to(m)

    for f in toate_focarele.get(zid, [])[:5]:
        folium.Marker([f["lat"], f["lon"]],
                      tooltip=f"Focar · {f['puncte']} puncte · {f['tendinta']}",
                      icon=folium.Icon(color="red", icon="fire")).add_to(m)

    det = list((z.get("detections") or {}).values())
    det.sort(key=lambda d: (d["date"], d["time"]), reverse=True)
    for d in det[:max_puncte]:
        t = cand(d)
        h = (datetime.now(timezone.utc) - t).total_seconds() / 3600 if t else 1e9
        culoare = "#e5484d" if h <= 6 else "#f76b15" if h <= 24 else "#f5d90a" if h <= 72 else "#6b7684"
        folium.CircleMarker(
            [d["lat"], d["lon"]],
            radius=max(3, min(18, 3 + (d.get("frp") or 0) ** 0.5 * 1.6)),
            color="#22d3ee" if d.get("src") == "ESA" else "#ffffff", weight=1.4,
            fill=True, fillColor=culoare, fillOpacity=0.6,
            popup=f"{d.get('sensor')}<br>FRP {d.get('frp')} MW<br>{d['date']} {d['time']}Z"
        ).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


for zid, z in zone.items():
    print(z.get("name", zid))
    display(harta(zid, z))

## 5. Suprafața arsă

Estimarea nu vine din numărarea punctelor, ci din **diferența de vegetație** măsurată de
Sentinel-2 înainte și după incendiu:

```
NBR  = (NIR − SWIR2) / (NIR + SWIR2)      benzile B8A și B12, la 20 m
dNBR = NBR_înainte − NBR_după
```

Din dNBR se scade apoi **medianul zonei de fundal**. Fără corecția asta cifra iese umflată:
seceta din regiune scade indicele de vegetație peste tot, iar dNBR brut raportează „ars"
pe suprafețe care doar s-au uscat.

In [ ]:
print(f"{'zonă':<34}{'arsă ≥0,27':>12}{'≥0,10':>10}{'sever':>9}   scene")
print("-" * 96)
for zid, z in zone.items():
    b = z.get("burn") or {}
    if not b:
        print(f"{z.get('name', zid):<34}{'se calculează':>12}")
        continue
    print(f"{z.get('name', zid):<34}{b.get('ha_moderat_plus', 0):>10.0f} ha"
          f"{b.get('ha_total', 0):>10.0f} ha{b.get('ha_sever', 0):>7.1f} ha   "
          f"{b.get('post_date')} față de {b.get('base_date')}")
    if b.get("background_dnbr") is not None:
        print(f"{'':<34}corecția de secetă aplicată: −{b['background_dnbr']:.3f} dNBR")

print("\nClase de severitate, pe zone:")
for zid, z in zone.items():
    b = z.get("burn") or {}
    if b.get("clase"):
        print(f"  {z.get('name', zid)}")
        for c in b["clase"]:
            print(f"    {c['clasa']:<20} prag ≥{c['prag']:<5} {c['ha']:>8.1f} ha")

print("\nATENȚIE: este o estimare proprie, NU o delimitare oficială.")
print("Clasa 0,10–0,27 e dominată de uscarea sezonieră și NU se folosește la raportare.")

## 6. Meteo, pericol și direcția de propagare

Indicele de pericol folosit este **Angström**, formă clasică și verificabilă din datele
brute care se afișează alături:

```
I = (H / 20) + (27 − T) / 10        H = umiditate %, T = temperatură °C
I < 2  foarte ridicat · 2–2,5 ridicat · 2,5–3 moderat · 3–4 scăzut · > 4 fără pericol
```

**Nu** folosim indicele canadian FWI: nu e disponibil pe API-ul gratuit, iar o implementare
parțială ar produce un număr care pare oficial fără să fie.

In [ ]:
for zid, z in zone.items():
    w = z.get("weather") or {}
    if not w:
        print(f"{z.get('name', zid)}: fără date meteo")
        continue
    d = w.get("danger") or {}
    t = w.get("terrain") or {}
    I = (w.get("humidity_pct", 0) or 0) / 20 + (27 - (w.get("temp_c", 0) or 0)) / 10

    print(f"\n=== {z.get('name', zid)} ===")
    print(f"  {w.get('temp_c')} °C · {w.get('humidity_pct')} % umiditate · "
          f"vânt {w.get('wind_kmh')} km/h (rafale {w.get('gust_kmh')})")
    print(f"  precipitații în 48 h: {w.get('precip_next48_mm')} mm   ·  VPD {w.get('vpd_kpa')} kPa")
    print(f"  indice Angström recalculat aici: {I:.2f}  (afișat: {d.get('index')}) "
          f"→ pericol {d.get('class')}")
    print(f"  vântul vine din {w.get('wind_from_compass')} ({w.get('wind_from_deg')}°) "
          f"→ pana de fum merge spre {w.get('downwind_compass')} ({w.get('downwind_deg')}°)")
    if t:
        if t.get("on_summit"):
            print(f"  relief {t.get('elevation_m')} m — focarul e pe vârf/creastă, deci panta "
                  f"NU determină direcția de propagare; vântul rămâne factorul principal")
        else:
            print(f"  relief {t.get('elevation_m')} m, pantă {t.get('slope_pct')}% "
                  f"spre {t.get('upslope_compass')}")

## 7. Verificare încrucișată: NASA vs ESA

Două sisteme independente măsoară același foc. Dacă arată amândouă același vârf de
intensitate, concluzia e solidă. Dacă nu, e un semn de întrebare.

In [ ]:
for zid, z in zone.items():
    zile = (z.get("stats") or {}).get("days") or []
    if len(zile) < 2:
        continue
    d = pd.DataFrame(zile)
    vn = d.loc[d["frp_nasa"].idxmax()]
    ve = d.loc[d["frp_esa"].idxmax()]
    print(f"\n{z.get('name', zid)}")
    print(f"  vârf NASA: {vn['date']} cu {vn['frp_nasa']:.0f} MW")
    print(f"  vârf ESA:  {ve['date']} cu {ve['frp_esa']:.0f} MW")
    if vn["date"] == ve["date"]:
        print("  → ambele surse indică aceeași zi: concluzie solidă")
    elif abs((pd.to_datetime(vn['date']) - pd.to_datetime(ve['date'])).days) <= 1:
        print("  → zile vecine: practic aceeași perioadă de vârf")
    else:
        print("  → vârfuri în zile diferite: de verificat, posibil nor sau trecere lipsă")

## 8. Ce NU putem stabili din aceste date

Onestitatea limitează concluziile. Din datele de aici **nu** rezultă:

- **Cauza** incendiului. Detecția termică arată unde arde, nu de ce.
- **Numărul exact de focare active** într-o anumită zi. Un focar poate produce mai multe
  puncte, iar un punct poate acoperi o zonă mai mare decât pixelul.
- **Suprafața oficială** afectată. Estimarea de aici e independentă și orientativă.
- **Separarea completă** a fumului de norul orografic. Am folosit clasificarea scenelor
(Sen2Cor) pentru asta, nu o analiză spectrală completă.

Ce **putem** spune cu tărie: unde s-a detectat activitate termică, când, cât de intensă,
și în ce direcție se propagă pana de fum în condițiile meteo actuale.

### Surse

- NASA FIRMS — `firms.modaps.eosdis.nasa.gov/data/active_fire/`
- Copernicus Data Space Ecosystem — produsele `SENTINEL-3/SLSTR/SL_2_FRP` și `SENTINEL-2/MSI/L2A`
- Open-Meteo — `api.open-meteo.com`
- Cod și metodă: <https://github.com/bunilastelian/domogled-live>